In [1]:
import os
import json
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import vstack

In [2]:
data_file = ('C:/Users/007pe/Downloads/dataset_preprocessed.json')
df = pd.read_json(data_file, lines=True)

In [3]:
print(df)

       Speaker_party_name                                               text  \
0        Liberal Democrat  1. What progress her Department has made on im...   
1            Conservative  The Government are on track to deliver their c...   
2        Liberal Democrat  It is clear that exit checks, which were scrap...   
3            Conservative  As I indicated in my original answer, we are o...   
4                  Labour  19. Given the situation at our border in Calai...   
...                   ...                                                ...   
670907       Conservative  I will have to check that point for the noble ...   
670908             Labour  My Lords, the Minister has referred a number o...   
670909       Conservative  The noble Lord is right that different aspects...   
670910             Labour  I thank noble Lords for a very interesting deb...   
670911       Conservative  My Lords, I wish the six or so stoic noble Pee...   

                                       

In [4]:
df = df[df['tokens'].apply(lambda x: len(x) > 10 and x is not None)]
df = df.drop(['text'], axis=1)

In [5]:
tfidf = TfidfVectorizer( 
    max_features=1000,
    stop_words='english',
    max_df=0.95,
    min_df=2,
    lowercase=False,         # Preserve case of your tokens (set to True if you want lowercase)
    preprocessor=lambda x: x,  # Bypass preprocessing
    tokenizer=lambda x: x.split()
)

In [6]:
tfidf_matrices = []
processed_chunks = []
chunk_size = 10000

# Process dataset in chunks
for i in range(0, len(df), chunk_size):
    print(f"Processing chunk {i} to {min(i + chunk_size, len(df))}...")
    
    # Get chunk and make a copy
    chunk = df.iloc[i:i + chunk_size].copy()

    texts = chunk['tokens'].apply(lambda tokens: ' '.join(tokens))
    
    # First chunk: fit and transform
    if i == 0:
        tfidf_matrix = tfidf.fit_transform(texts)
    # Subsequent chunks: transform only
    else:
        tfidf_matrix = tfidf.transform(texts)
    
    # Store the TF-IDF matrix
    tfidf_matrices.append(tfidf_matrix)
    
    # Optimize TF-IDF feature addition[poiuytrewqqw56789o[\789+=-0-+

    chunk['embedding'] = [tfidf_matrix[j].toarray()[0] for j in range(tfidf_matrix.shape[0])]
    
    processed_chunks.append(chunk)

# After all chunks are processed, combine results
final_df = pd.concat(processed_chunks, ignore_index=True)
final_tfidf_matrix = vstack(tfidf_matrices)


# Now that processing is complete, you can access results
print(f"Processed DataFrame shape: {final_df.shape}")
print(f"TF-IDF Matrix shape: {final_tfidf_matrix.shape}")
print("\nSample of processed DataFrame:")
print(final_df.head())

# Access feature names
feature_names = tfidf.get_feature_names_out()
print(f"\nNumber of features: {len(feature_names)}")
print(f"Sample features: {feature_names[:5]}")

Processing chunk 0 to 10000...


C:\Users\007pe\PycharmProjects\embeddings\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Processing chunk 10000 to 20000...
Processing chunk 20000 to 30000...
Processing chunk 30000 to 40000...
Processing chunk 40000 to 50000...
Processing chunk 50000 to 60000...
Processing chunk 60000 to 70000...
Processing chunk 70000 to 80000...
Processing chunk 80000 to 90000...
Processing chunk 90000 to 100000...
Processing chunk 100000 to 110000...
Processing chunk 110000 to 120000...
Processing chunk 120000 to 130000...
Processing chunk 130000 to 140000...
Processing chunk 140000 to 150000...
Processing chunk 150000 to 160000...
Processing chunk 160000 to 170000...
Processing chunk 170000 to 180000...
Processing chunk 180000 to 190000...
Processing chunk 190000 to 200000...
Processing chunk 200000 to 210000...
Processing chunk 210000 to 220000...
Processing chunk 220000 to 230000...
Processing chunk 230000 to 240000...
Processing chunk 240000 to 250000...
Processing chunk 250000 to 260000...
Processing chunk 260000 to 270000...
Processing chunk 270000 to 280000...
Processing chunk 2

In [7]:
from collections import defaultdict

party_ideology_mapping = {
    'Conservative': 'Centre-right to right-wing',
    'Labour': 'Centre-left',
    'Liberal Democrat': 'Centre to centre-left',
    'Scottish National Party': 'Centre-left to left-wing',
    'Crossbench': 'Non-partisan',
    'Democratic Unionist Party': 'Right-wing',
    'Green Party': 'Left-wing',
    'Plaid Cymru': 'Centre-left to left-wing',
    'Independent': 'Independent',
    'Bishops': 'Cleric',
    'Social Democratic & Labour Party': 'Centre-left',
    'Non-affiliated': 'Non-partisan',
    'Independent;Conservative': 'Centre-right to right-wing',
    'Labour;Non-affiliated': 'Centre-left',
    'Ulster Unionist Party': 'Right-wing',
    'UK Independence Party': 'Right-wing to far-right',
    'Alliance': 'Centre to centre-left',
    'Non-affiliated;Crossbench': 'Non-partisan',
    'Scottish National Party;Independent': 'Centre-left to left-wing',
    'Non-affiliated;Conservative': 'Centre-right to right-wing',
    'Non-affiliated;Labour': 'Centre-left',
    'Conservative;Independent': 'Centre-right to right-wing',
    'Plaid Cymru;Independent': 'Centre-left to left-wing',
    'Conservative;Crossbench': 'Centre-right to right-wing',
    'Scottish National Party;Alba Party': 'Centre-left to left-wing',
    'The Independent Group for Change': 'Centre-left',
    'Independent Labour': 'Independent',
    'Labour Independent': 'Centre-left',
    'Conservative;Non-affiliated': 'Centre-right to right-wing',
    'Labour;Independent': 'Centre-left',
    'Liberal Democrat;Non-affiliated': 'Centre to centre-left',
    'Change UK - The Independent Group;The Independent Group for Change': 'Centre-left',
    'Independent;Liberal Democrat': 'Centre to centre-left',
    'Liberal Democrat;Crossbench': 'Centre to centre-left',
    'Independent Ulster Unionist': 'Independent',
    'Independent Liberal Democrat': 'Independent',
    'Independent;Crossbench': 'Independent',
    'Conservative;Labour': 'Centre-right to right-wing',  # Rare combination, likely an error
    'Independent Social Democrat': 'Independent',
    'Respect': 'Left-wing to far-left',
    'The Independent Group for Change;Change UK - The Independent Group': 'Centre-left',
    'UK Independence Party;Non-affiliated': 'Right-wing to far-right',
    'Independent Labour;Non-affiliated': 'Centre-left',
    'Conservative;Independent;Liberal Democrat': 'Centre-right to right-wing',
    'Conservative;Liberal Democrat': 'Centre-right to right-wing',  # Rare combination, likely an error
    'Conservative;Non-affiliated;Crossbench': 'Centre-right to right-wing',
    'Conservative;Independent Conservative;Non-affiliated': 'Centre-right to right-wing',
    'UK Independence Party;Independent': 'Right-wing to far-right',
    'Liberal Democrat;Non-affiliated;Crossbench': 'Centre to centre-left',
    'Crossbench;Labour': 'Non-partisan'
}

mapped_dict = defaultdict(set)

for party, ideology in party_ideology_mapping.items():
    mapped_dict[ideology].update([party])

def map_values(category):
    for ideology, parties in mapped_dict.items():
        if category in parties:
            return ideology
    return None

final_df['Speaker_party_name'] = final_df['Speaker_party_name'].apply(map_values)
print(final_df)
print(final_df['Speaker_party_name'].value_counts())

                Speaker_party_name  \
0       Centre-right to right-wing   
1            Centre to centre-left   
2       Centre-right to right-wing   
3                      Centre-left   
4       Centre-right to right-wing   
...                            ...   
591683                 Centre-left   
591684  Centre-right to right-wing   
591685                 Centre-left   
591686  Centre-right to right-wing   
591687                 Centre-left   

                                                   tokens  \
0       [government, track, deliver, commitment, intro...   
1       [clear, exit, check, scrap, previous, labour, ...   
2       [indicate, original, answer, track, ensure, ex...   
3       [give, situation, border, calais, home, secret...   
4       [great, deal, work, french, authority, relatio...   
...                                                   ...   
591683  [argument, law, protect, everybody, action, ta...   
591684  [check, point, noble, lord, write, memory, act.

In [8]:
final_df.to_json('C:/Users/007pe/Downloads/tdidf_embeddings.json', orient='records', lines=True)